In [1]:
"""
Sentinel RQ1 Analysis — Phase 3 of INFO 5001 Research Project

Extends Virkud et al.'s RQ1 analysis by adding Microsoft Sentinel
as a 5th ruleset. Tests the pre-registered hypothesis that
Sentinel over-covers cloud/identity techniques and under-covers
host-only techniques.

Author: Junaid Abrar Razeen
"""

import os
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

# Local helpers
from sentinel_adapter import (
    load_sentinel,
    technique_coverage_summary,
    techniques_unique_to_sentinel,
    test_prereg_hypothesis,
    coverage_percent_per_technique,
    compute_technique_counts,
    PREREG_CLOUD_TECHNIQUES,
    PREREG_HOST_TECHNIQUES,
)
from virkud_loader import load_all_virkud

# Paths
SENTINEL_CSV = os.path.join('..', 'sentinel_data', 'MicrosoftSentinel.csv')
VIRKUD_DATA = os.path.join('..', '..', 'data')

# Pretty pandas
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

In [2]:
# Load Virkud's three publicly-available rulesets
techniques, splunk, elastic, sigma = load_all_virkud(VIRKUD_DATA)

# Load Sentinel via our adapter
sentinel = load_sentinel(SENTINEL_CSV)

print('\n=== All rulesets loaded ===')
print(f'  Splunk:   {len(splunk):>5} rules')
print(f'  Elastic:  {len(elastic):>5} rules')
print(f'  Sigma:    {len(sigma):>5} rules')
print(f'  Sentinel: {len(sentinel):>5} rules')

[virkud] Loaded 578 technique-tactic rows
[virkud] Splunk: 911 rules
[virkud] Elastic: 473 rules
[virkud] Sigma: 2195 rules
[adapter] Loaded 10132 raw rows from ..\sentinel_data\MicrosoftSentinel.csv
[adapter] After TechniqueId filter: 6810 rows
[adapter] After Analytics filter: 3107 rows
[adapter] After deduplication: 349 unique rules
[adapter] Final dataframe: 349 rules
[adapter] Unique techniques: 81

=== All rulesets loaded ===
  Splunk:     911 rules
  Elastic:    473 rules
  Sigma:     2195 rules
  Sentinel:   349 rules


In [3]:
# Build the headline coverage comparison
summary, coverage_sets = technique_coverage_summary(
    sentinel, splunk, elastic, sigma, techniques
)
print('=== Technique Coverage Summary ===')
print(summary.to_string())

=== Technique Coverage Summary ===
          # Rules  # Unique Techniques % Coverage (ATT&CK v11)
Splunk        911                  100                   52.4%
Elastic       473                   92                   48.2%
Sigma        2195                  151                   79.1%
Sentinel      349                   79                   41.4%


In [4]:
 # Find what Sentinel covers that the on-host EDRs don't
deltas = techniques_unique_to_sentinel(coverage_sets)

print('=== Techniques unique to Sentinel (not in Splunk or Elastic) ===')
unique_to_sentinel = sorted(deltas['sentinel_only_vs_commercial'])
print(f'Count: {len(unique_to_sentinel)}')
print(f'Techniques: {unique_to_sentinel}')

print('\n=== Techniques in Splunk/Elastic but NOT in Sentinel ===')
missed_by_sentinel = sorted(deltas['covered_by_others_not_sentinel'])
print(f'Count: {len(missed_by_sentinel)}')
print(f'Techniques: {missed_by_sentinel}')

=== Techniques unique to Sentinel (not in Splunk or Elastic) ===
Count: 12
Techniques: ['T1008', 'T1012', 'T1030', 'T1041', 'T1052', 'T1072', 'T1213', 'T1496', 'T1528', 'T1563', 'T1571', 'T1578']

=== Techniques in Splunk/Elastic but NOT in Sentinel ===
Count: 57
Techniques: ['T1001', 'T1006', 'T1014', 'T1016', 'T1033', 'T1037', 'T1047', 'T1049', 'T1056', 'T1057', 'T1069', 'T1082', 'T1083', 'T1095', 'T1106', 'T1112', 'T1113', 'T1115', 'T1120', 'T1123', 'T1124', 'T1127', 'T1129', 'T1135', 'T1187', 'T1197', 'T1201', 'T1202', 'T1218', 'T1219', 'T1220', 'T1222', 'T1482', 'T1484', 'T1490', 'T1491', 'T1497', 'T1518', 'T1526', 'T1529', 'T1535', 'T1537', 'T1539', 'T1553', 'T1555', 'T1560', 'T1565', 'T1580', 'T1587', 'T1588', 'T1589', 'T1590', 'T1592', 'T1595', 'T1614', 'T1621', 'T1647']


In [5]:
# Test the pre-registered hypothesis
hypothesis_df = test_prereg_hypothesis(sentinel, splunk, elastic, sigma)
print('=== Pre-Registered Hypothesis Test ===')
print('Hypothesis: Sentinel over-covers cloud/identity, under-covers host-only.\n')
print(hypothesis_df.to_string(index=False))

=== Pre-Registered Hypothesis Test ===
Hypothesis: Sentinel over-covers cloud/identity, under-covers host-only.

Technique                                  Name                Category  Splunk  Elastic  Sigma  Sentinel
    T1078                        Valid Accounts Cloud/Identity (prereg)      39       13     52        78
    T1098                  Account Manipulation Cloud/Identity (prereg)       6        7     23        38
    T1530               Data from Cloud Storage Cloud/Identity (prereg)       2        0      0         1
    T1110 Brute Force (incl. password spraying) Cloud/Identity (prereg)      15       11     22        29
    T1538               Cloud Service Dashboard Cloud/Identity (prereg)       0        0      0         0
    T1526               Cloud Service Discovery Cloud/Identity (prereg)       4        0      1         0
    T1580        Cloud Infrastructure Discovery Cloud/Identity (prereg)       2        0      0         0
    T1136                        Create

In [6]:
# Compute coverage % per technique for each ruleset
splunk_pct = coverage_percent_per_technique(splunk, techniques)
elastic_pct = coverage_percent_per_technique(elastic, techniques)
sigma_pct = coverage_percent_per_technique(sigma, techniques)
sentinel_pct = coverage_percent_per_technique(sentinel, techniques)

# Align all on the full ATT&CK v11 technique index
all_techs = pd.Index(techniques['technique'].drop_duplicates().values)

splunk_aligned = splunk_pct.reindex(all_techs, fill_value=0)
elastic_aligned = elastic_pct.reindex(all_techs, fill_value=0)
sigma_aligned = sigma_pct.reindex(all_techs, fill_value=0)
sentinel_aligned = sentinel_pct.reindex(all_techs, fill_value=0)

print('=== Spearman Rank Correlations (Sentinel vs other rulesets) ===')
for other_name, other_aligned in [('Splunk', splunk_aligned),
                                   ('Elastic', elastic_aligned),
                                   ('Sigma', sigma_aligned)]:
    result = stats.spearmanr(sentinel_aligned.values, other_aligned.values)
    print(f'  Sentinel  vs  {other_name:>7}:  '
          f'ρ = {result.statistic:.4f}, p = {result.pvalue:.2e}')

print('\n=== Spearman within Virkud (replication for sanity) ===')
for a_name, a_aligned, b_name, b_aligned in [
    ('Splunk',  splunk_aligned,  'Elastic', elastic_aligned),
    ('Splunk',  splunk_aligned,  'Sigma',   sigma_aligned),
    ('Elastic', elastic_aligned, 'Sigma',   sigma_aligned),
]:
    result = stats.spearmanr(a_aligned.values, b_aligned.values)
    print(f'  {a_name:>7}  vs  {b_name:>7}:  '
          f'ρ = {result.statistic:.4f}, p = {result.pvalue:.2e}')

=== Spearman Rank Correlations (Sentinel vs other rulesets) ===
  Sentinel  vs   Splunk:  ρ = 0.3927, p = 1.92e-08
  Sentinel  vs  Elastic:  ρ = 0.4461, p = 9.93e-11
  Sentinel  vs    Sigma:  ρ = 0.5003, p = 1.72e-13

=== Spearman within Virkud (replication for sanity) ===
   Splunk  vs  Elastic:  ρ = 0.6389, p = 2.69e-23
   Splunk  vs    Sigma:  ρ = 0.7010, p = 1.47e-29
  Elastic  vs    Sigma:  ρ = 0.7617, p = 1.82e-37


In [7]:
# Persist the headline outputs for use in the report write-up
output_dir = os.path.join('..', 'outputs')
os.makedirs(output_dir, exist_ok=True)

summary.to_csv(os.path.join(output_dir, 'coverage_summary.csv'))
hypothesis_df.to_csv(os.path.join(output_dir, 'hypothesis_test.csv'), index=False)

with open(os.path.join(output_dir, 'sentinel_unique_techniques.txt'), 'w') as f:
    f.write('Techniques unique to Sentinel (not in Splunk or Elastic):\n')
    for t in sorted(deltas['sentinel_only_vs_commercial']):
        f.write(f'  {t}\n')
    f.write('\nTechniques in Splunk/Elastic but not in Sentinel:\n')
    for t in sorted(deltas['covered_by_others_not_sentinel']):
        f.write(f'  {t}\n')

print('Saved outputs to extension/outputs/')

Saved outputs to extension/outputs/
